In [ ]:
!nvidia-smi
!pip -q install kagglehub


In [ ]:
%cd /content
!rm -rf yolov11s_kuangjing
!git clone https://github.com/tuanziya666/yolov11s_kuangjing.git
%cd /content/yolov11s_kuangjing
!git checkout main
!git pull


In [ ]:
!python -c "from ultralytics import YOLO; YOLO('ultralytics/cfg/models/11/yolo11s-SPDConv.yaml'); print('SPDConv parse ok')"
!ls -lh train_yolo11s_spdconv_atfl.py
!ls -lh ultralytics/cfg/models/11/yolo11s-SPDConv.yaml


In [ ]:
from pathlib import Path
import shutil
import zipfile
import kagglehub

KAGGLE_DATASET = 'yuanfangshang/yolo-kuangjing-hard15k-dataset'  # 改成你的真实 slug
download_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print('Downloaded to:', download_path)

def find_dataset_root(base: Path):
    if (base / 'images' / 'train').exists() and (base / 'labels' / 'train').exists():
        return base
    for p in base.rglob('*'):
        if p.is_dir() and (p / 'images' / 'train').exists() and (p / 'labels' / 'train').exists():
            return p
    return None

DATASET_ROOT = find_dataset_root(download_path)

if DATASET_ROOT is None:
    zips = list(download_path.rglob('*.zip'))
    if not zips:
        raise FileNotFoundError('没找到可用数据集目录或 zip，请检查 Kaggle 数据集内容。')
    extract_dir = Path('/content/dataset_unzip')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0], 'r') as zf:
        zf.extractall(extract_dir)
    DATASET_ROOT = find_dataset_root(extract_dir)

if DATASET_ROOT is None:
    raise FileNotFoundError('解压后仍然没找到 images/train 和 labels/train。')

print('Resolved DATASET_ROOT =', DATASET_ROOT)


In [ ]:
from pathlib import Path

yaml_text = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
"""

yaml_path = Path('/content/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_colab.yaml')
yaml_path.write_text(yaml_text, encoding='utf-8')
print(yaml_text)


In [ ]:
%cd /content/yolov11s_kuangjing

RUN_NAME = 'yolo11s_spdconv_atfl_colab'
EPOCHS = 100
BATCH = 16
WORKERS = 2
IMGSZ = 640
SEED = 42

!python -u train_yolo11s_spdconv_atfl.py \
  --data ultralytics/cfg/datasets/coalmine4_colab.yaml \
  --name $RUN_NAME \
  --project /content/runs \
  --device 0 \
  --epochs $EPOCHS \
  --batch $BATCH \
  --workers $WORKERS \
  --imgsz $IMGSZ \
  --seed $SEED


In [ ]:
%cd /content/yolov11s_kuangjing

from ultralytics import YOLO

BEST = f'/content/runs/{RUN_NAME}/weights/best.pt'
print('BEST =', BEST)

model = YOLO(BEST)

model.val(
    data='ultralytics/cfg/datasets/coalmine4_colab.yaml',
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project='/content/evals',
    name=f'{RUN_NAME}_val',
    exist_ok=True,
)

model.val(
    data='ultralytics/cfg/datasets/coalmine4_colab.yaml',
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project='/content/evals',
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)


In [ ]:
%cd /content
!zip -qr yolo11s_spdconv_atfl_colab_results.zip runs/yolo11s_spdconv_atfl_colab evals/yolo11s_spdconv_atfl_colab_val evals/yolo11s_spdconv_atfl_colab_test
!ls -lh yolo11s_spdconv_atfl_colab_results.zip
